# SAGAR DRISHTI — Global GRU iceberg trajectory model

Research notebook that trains the **base forecasting model** used by the platform:
a global GRU that reads **14 consecutive daily positions** of an Antarctic iceberg
and predicts its **next 7 daily positions** in one pass, plus the **p90 error radius**
per forecast day (the "risk radius" used for forecast cones and, later, route-planning
danger zones).

| | |
|---|---|
| Data | BYU MERS Consolidated Antarctic Iceberg Database v8.0 (`consolidated_database_v8.0.zip`) |
| Projection | EPSG:3031 Antarctic Polar Stereographic (metres) |
| Input | 14 × 6 features: `relative_x_km, relative_y_km, vx_km_per_day, vy_km_per_day, season_sin, season_cos` |
| Output | 14 values: D+1…D+7 (x, y) displacement in km from the last input position |
| Split | chronological by anchor date, 75 / 12.5 / 12.5 — no shuffling across time |
| Produces | `global_gru_trajectory_model.keras`, `global_gru_trajectory_scalers.joblib`, `metadata.json`, `risk_radius_km_p90.json` |

**Using the result in the platform:** copy the three files from the exported
`sagar_drishti_base_model` folder into `models/base/` and run `dev bootstrap`.
Production code does not import this notebook — the same steps live in the `ml/`
package (`ml/adapters/bootstrap_adapter.py`, `ml/training/*`, `ml/models/gru_architecture.py`).

Run on Colab with a GPU runtime (≈10 min), or locally from the `notebooks/` folder.

## 1. Setup and data

In [ ]:
!pip -q install pyproj

In [ ]:
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from pyproj import Transformer

ZIP_NAME = "consolidated_database_v8.0.zip"

try:
    from google.colab import files  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ZIP_PATH = Path("/content") / ZIP_NAME
    if not ZIP_PATH.exists():
        print(f"Upload {ZIP_NAME}")
        files.upload()
    WORK_DIR = Path("/content")
else:
    ZIP_PATH = Path("../data/bootstrap") / ZIP_NAME
    WORK_DIR = Path(os.environ.get("SAGAR_NOTEBOOK_WORKDIR", "../data/processed/notebook"))

WORK_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR = WORK_DIR / "byu_iceberg_raw"
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(CSV_DIR)

print("Dataset:", ZIP_PATH)
print("Iceberg CSV files:", len(list(CSV_DIR.rglob("*.csv"))))

## 2. Clean trajectories

One position per iceberg per day. Each BYU file holds one iceberg; positions come
from several sensors, so the first valid source in `SOURCE_PRIORITY` is used.
`0/0` placeholders are excluded by the latitude/longitude range check. Positions are
projected to EPSG:3031 metres.

In [ ]:
SOURCE_PRIORITY = ["nic", "qscat", "ers", "ascat", "seawinds", "nscat", "oscat"]

all_tracks = []
for csv_path in sorted(CSV_DIR.rglob("*.csv")):
    raw = pd.read_csv(csv_path)
    if "date" not in raw.columns:
        continue

    clean = pd.DataFrame(index=raw.index)
    clean["iceberg_id"] = csv_path.stem.upper()  # a23a.csv -> A23A

    # YYYYDDD date code -> date
    date_code = pd.to_numeric(raw["date"], errors="coerce")
    year = (date_code // 1000).astype("Int64")
    day_of_year = (date_code % 1000).astype("Int64")
    clean["date"] = pd.to_datetime(
        year.astype("string") + day_of_year.astype("string").str.zfill(3),
        format="%Y%j",
        errors="coerce",
    )

    clean["latitude"] = np.nan
    clean["longitude"] = np.nan
    clean["position_source"] = pd.NA
    for source in SOURCE_PRIORITY:
        lat_col, lon_col = f"{source}_1", f"{source}_2"
        if lat_col not in raw.columns or lon_col not in raw.columns:
            continue
        lat = pd.to_numeric(raw[lat_col], errors="coerce")
        lon = pd.to_numeric(raw[lon_col], errors="coerce")
        choose = clean["latitude"].isna() & lat.between(-90, -45) & lon.between(-180, 180)
        clean.loc[choose, "latitude"] = lat[choose]
        clean.loc[choose, "longitude"] = lon[choose]
        clean.loc[choose, "position_source"] = source

    clean = clean.dropna(subset=["date", "latitude", "longitude"])
    if not clean.empty:
        all_tracks.append(clean)

tracks = (
    pd.concat(all_tracks, ignore_index=True)
    .sort_values(["iceberg_id", "date"])
    .drop_duplicates(["iceberg_id", "date"], keep="first")
    .reset_index(drop=True)
)

to_antarctic = Transformer.from_crs("EPSG:4326", "EPSG:3031", always_xy=True)
tracks["x_m"], tracks["y_m"] = to_antarctic.transform(tracks["longitude"].to_numpy(), tracks["latitude"].to_numpy())

print("Clean trajectory rows:", len(tracks))
print("Icebergs:", tracks["iceberg_id"].nunique())
print("Date range:", tracks["date"].min().date(), "to", tracks["date"].max().date())
print(tracks["position_source"].value_counts())

## 3. Build 14-day → 7-day sequences

Windows are taken only inside uninterrupted **daily** runs of each track.
Features are relative to the last input day (the anchor); the target is the
displacement (km) from the anchor for days 1–7.

In [ ]:
try:
    from tqdm.auto import tqdm
except ImportError:  # progress bar is optional
    def tqdm(iterable, **_):
        return iterable

HISTORY_DAYS = 14
FORECAST_DAYS = 7
FEATURE_NAMES = ["relative_x_km", "relative_y_km", "vx_km_per_day", "vy_km_per_day", "season_sin", "season_cos"]

sequence_features, sequence_targets = [], []
anchor_dates, anchor_x_m, anchor_y_m = [], [], []
last_vx_km_per_day, last_vy_km_per_day = [], []

for iceberg_id, group in tqdm(tracks.groupby("iceberg_id"), total=tracks["iceberg_id"].nunique(), desc="Sequences"):
    group = group.sort_values("date").reset_index(drop=True)
    run_id = group["date"].diff().dt.days.ne(1).cumsum()  # new run wherever daily continuity breaks

    for _, run in group.groupby(run_id):
        run = run.reset_index(drop=True)
        if len(run) < HISTORY_DAYS + FORECAST_DAYS:
            continue

        x_values = run["x_m"].to_numpy(dtype=np.float64)
        y_values = run["y_m"].to_numpy(dtype=np.float64)
        date_values = run["date"].to_numpy(dtype="datetime64[ns]")
        day_of_year = run["date"].dt.dayofyear.to_numpy(dtype=np.float64)

        for end in range(HISTORY_DAYS - 1, len(run) - FORECAST_DAYS):
            start = end - HISTORY_DAYS + 1
            hist_x, hist_y = x_values[start:end + 1], y_values[start:end + 1]
            current_x, current_y = hist_x[-1], hist_y[-1]

            vx = np.empty(HISTORY_DAYS)
            vy = np.empty(HISTORY_DAYS)
            vx[1:] = np.diff(hist_x) / 1000.0  # daily run: 1 day between entries
            vy[1:] = np.diff(hist_y) / 1000.0
            vx[0], vy[0] = vx[1], vy[1]

            season_angle = 2 * np.pi * day_of_year[start:end + 1] / 365.25
            features = np.column_stack([
                (hist_x - current_x) / 1000.0,
                (hist_y - current_y) / 1000.0,
                vx,
                vy,
                np.sin(season_angle),
                np.cos(season_angle),
            ])

            future_x = x_values[end + 1:end + 1 + FORECAST_DAYS]
            future_y = y_values[end + 1:end + 1 + FORECAST_DAYS]
            target = np.column_stack([(future_x - current_x) / 1000.0, (future_y - current_y) / 1000.0])

            sequence_features.append(features.astype(np.float32))
            sequence_targets.append(target.reshape(-1).astype(np.float32))
            anchor_dates.append(date_values[end])
            anchor_x_m.append(current_x)
            anchor_y_m.append(current_y)
            last_vx_km_per_day.append(vx[-1])
            last_vy_km_per_day.append(vy[-1])

X_sequences = np.stack(sequence_features)
y_sequences = np.stack(sequence_targets)
anchor_dates = np.asarray(anchor_dates, dtype="datetime64[ns]")
anchor_x_m = np.asarray(anchor_x_m)
anchor_y_m = np.asarray(anchor_y_m)
last_vx_km_per_day = np.asarray(last_vx_km_per_day)
last_vy_km_per_day = np.asarray(last_vy_km_per_day)

print("Input shape:", X_sequences.shape, "| target shape:", y_sequences.shape)

## 4. Chronological split and scalers

All samples from the same anchor date stay in the same phase, so no information
from a test date leaks into training. Scalers are fitted on the training phase only.

In [ ]:
from sklearn.preprocessing import StandardScaler

TOTAL_EXAMPLES = len(X_sequences)
cumulative_counts = pd.Series(anchor_dates).value_counts().sort_index().cumsum()
train_end_date = cumulative_counts.index[np.searchsorted(cumulative_counts.to_numpy(), TOTAL_EXAMPLES * 0.75)]
validation_end_date = cumulative_counts.index[np.searchsorted(cumulative_counts.to_numpy(), TOTAL_EXAMPLES * 0.875)]

train_mask = anchor_dates <= train_end_date
validation_mask = (anchor_dates > train_end_date) & (anchor_dates <= validation_end_date)
test_mask = anchor_dates > validation_end_date

X_train_raw, X_validation_raw, X_test_raw = X_sequences[train_mask], X_sequences[validation_mask], X_sequences[test_mask]
y_train_raw, y_validation_raw, y_test_raw = y_sequences[train_mask], y_sequences[validation_mask], y_sequences[test_mask]

feature_scaler = StandardScaler().fit(X_train_raw.reshape(-1, len(FEATURE_NAMES)))
target_scaler = StandardScaler().fit(y_train_raw)


def scale_features(X):
    return feature_scaler.transform(X.reshape(-1, len(FEATURE_NAMES))).reshape(X.shape).astype(np.float32)


X_train, X_validation, X_test = scale_features(X_train_raw), scale_features(X_validation_raw), scale_features(X_test_raw)
y_train = target_scaler.transform(y_train_raw).astype(np.float32)
y_validation = target_scaler.transform(y_validation_raw).astype(np.float32)
y_test = target_scaler.transform(y_test_raw).astype(np.float32)


def phase(name, mask):
    d = anchor_dates[mask]
    return {"Phase": name, "Examples": int(mask.sum()), "Percentage": 100 * mask.sum() / TOTAL_EXAMPLES,
            "From": str(pd.Timestamp(d.min()).date()), "To": str(pd.Timestamp(d.max()).date())}


split_summary = pd.DataFrame([phase("Training", train_mask), phase("Validation", validation_mask), phase("Testing", test_mask)])
display(split_summary.round(2))

## 5. Train the GRU

Architecture (must stay identical to `ml/models/gru_architecture.py`,
`architecture_version = gru_entry14_to_day7_v1`, 44,366 parameters):

`Input(14, 6) → GRU(96, dropout 0.10) → LayerNormalization → Dense(128, relu) → Dropout(0.15) → Dense(14)`

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import callbacks, layers

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

model = tf.keras.Sequential([
    layers.Input(shape=(HISTORY_DAYS, len(FEATURE_NAMES))),
    layers.GRU(96, dropout=0.10, name="trajectory_gru"),
    layers.LayerNormalization(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.15),
    layers.Dense(FORECAST_DAYS * 2, name="future_displacements"),  # D1 x/y ... D7 x/y
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4), loss=tf.keras.losses.Huber(), metrics=["mae"])
model.summary()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_validation, y_validation),
    epochs=65,  # upper bound; early stopping usually ends sooner
    batch_size=512,
    callbacks=[
        callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1),
    ],
    verbose=1,
)

plt.figure(figsize=(9, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Huber loss")
plt.grid(alpha=0.3)
plt.legend()
plt.show()
print("Epochs run:", len(history.history["loss"]), "| best epoch:", int(np.argmin(history.history["val_loss"])) + 1)

## 6. Test-set accuracy and p90 risk radius

Great-circle error (km) between predicted and actual positions on the unseen test
phase, for **every** forecast day D+1…D+7. The **90th-percentile error per day is the
risk radius**: in 90 % of test cases the iceberg was inside this distance of the
forecast point. The platform draws it as the forecast cone and it will define the
danger zone for route planning. Constant velocity is shown only as a reference
benchmark — it is not a production model.

In [ ]:
to_wgs84 = Transformer.from_crs("EPSG:3031", "EPSG:4326", always_xy=True)


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))


gru_prediction_km = target_scaler.inverse_transform(model.predict(X_test, batch_size=1024, verbose=1)).reshape(-1, FORECAST_DAYS, 2)
actual_future_km = y_test_raw.reshape(-1, FORECAST_DAYS, 2)
test_anchor_x_m, test_anchor_y_m = anchor_x_m[test_mask], anchor_y_m[test_mask]
test_vx, test_vy = last_vx_km_per_day[test_mask], last_vy_km_per_day[test_mask]


def errors_km(dx_km, dy_km, horizon_index):
    actual_lon, actual_lat = to_wgs84.transform(
        test_anchor_x_m + actual_future_km[:, horizon_index, 0] * 1000, test_anchor_y_m + actual_future_km[:, horizon_index, 1] * 1000
    )
    pred_lon, pred_lat = to_wgs84.transform(test_anchor_x_m + dx_km * 1000, test_anchor_y_m + dy_km * 1000)
    return haversine_km(actual_lat, actual_lon, pred_lat, pred_lon)


rows = []
for horizon in range(1, FORECAST_DAYS + 1):
    i = horizon - 1
    gru = errors_km(gru_prediction_km[:, i, 0], gru_prediction_km[:, i, 1], i)
    base = errors_km(test_vx * horizon, test_vy * horizon, i)
    rows.append({
        "Horizon (days)": horizon,
        "Test examples": len(gru),
        "GRU mean error (km)": gru.mean(),
        "GRU RMSE (km)": np.sqrt(np.mean(gru ** 2)),
        "GRU median error (km)": np.median(gru),
        "GRU p90 error (km)": np.quantile(gru, 0.90),
        "Baseline mean error (km)": base.mean(),
        "Baseline median error (km)": np.median(base),
        "Baseline p90 error (km)": np.quantile(base, 0.90),
    })

gru_test_metrics = pd.DataFrame(rows)
risk_radius_km_p90 = {int(r["Horizon (days)"]): round(float(r["GRU p90 error (km)"]), 3) for _, r in gru_test_metrics.iterrows()}

display(gru_test_metrics.round(3))
print("p90 risk radius by forecast day (km):", risk_radius_km_p90)

In [ ]:
x = np.arange(FORECAST_DAYS)
w = 0.38
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(x - w / 2, gru_test_metrics["Baseline mean error (km)"], w, label="Constant velocity (reference)")
axes[0].bar(x + w / 2, gru_test_metrics["GRU mean error (km)"], w, label="GRU")
axes[0].set_title("Mean positional error")
axes[1].bar(x - w / 2, gru_test_metrics["Baseline p90 error (km)"], w, label="Constant velocity (reference)")
axes[1].bar(x + w / 2, gru_test_metrics["GRU p90 error (km)"], w, label="GRU")
axes[1].set_title("p90 error = risk radius")
for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels([f"D+{h}" for h in range(1, FORECAST_DAYS + 1)])
    ax.set_ylabel("km")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()
plt.suptitle("GRU vs constant velocity on the unseen test period")
plt.tight_layout()
plt.show()

## 7. Example: 7-day forecasts with risk radius

Latest 14-day window of every iceberg that has one in this historical dataset.
These are **historical demonstrations**, not live forecasts — live forecasts are
produced by the platform from official USNIC observations.

In [ ]:
latest_inputs = []
for iceberg_id, track in tracks.groupby("iceberg_id"):
    track = track.sort_values("date").reset_index(drop=True)
    run_id = track["date"].diff().dt.days.ne(1).cumsum()
    history_rows = track[run_id == run_id.iloc[-1]].tail(HISTORY_DAYS)
    if len(history_rows) < HISTORY_DAYS:
        continue
    hx, hy = history_rows["x_m"].to_numpy(float), history_rows["y_m"].to_numpy(float)
    vx = np.empty(HISTORY_DAYS)
    vy = np.empty(HISTORY_DAYS)
    vx[1:], vy[1:] = np.diff(hx) / 1000.0, np.diff(hy) / 1000.0
    vx[0], vy[0] = vx[1], vy[1]
    angle = 2 * np.pi * history_rows["date"].dt.dayofyear.to_numpy(float) / 365.25
    latest_inputs.append({
        "iceberg_id": iceberg_id,
        "latest_date": history_rows["date"].iloc[-1],
        "x_m": hx[-1],
        "y_m": hy[-1],
        "features": np.column_stack([(hx - hx[-1]) / 1000, (hy - hy[-1]) / 1000, vx, vy, np.sin(angle), np.cos(angle)]),
    })

latest_km = target_scaler.inverse_transform(
    model.predict(scale_features(np.stack([item["features"] for item in latest_inputs]).astype(np.float32)), batch_size=512, verbose=0)
).reshape(-1, FORECAST_DAYS, 2)

forecast_rows = []
for item, disp in zip(latest_inputs, latest_km):
    for horizon in range(1, FORECAST_DAYS + 1):
        px, py = item["x_m"] + disp[horizon - 1, 0] * 1000, item["y_m"] + disp[horizon - 1, 1] * 1000
        lon, lat = to_wgs84.transform(px, py)
        forecast_rows.append({
            "iceberg_id": item["iceberg_id"],
            "latest_observation_date": item["latest_date"].date().isoformat(),
            "forecast_horizon_days": horizon,
            "forecast_date": (item["latest_date"] + pd.Timedelta(days=horizon)).date().isoformat(),
            "predicted_latitude": lat,
            "predicted_longitude": lon,
            "risk_radius_km_p90": risk_radius_km_p90[horizon],
            "forecast_status": "historical_demo_not_real_time",
        })

example_forecasts = pd.DataFrame(forecast_rows)
print("Icebergs forecast:", example_forecasts["iceberg_id"].nunique())
display(example_forecasts[example_forecasts["iceberg_id"] == "A23A"].round(4))

## 8. Export for the platform

Writes `sagar_drishti_base_model/` (and a zip) with exactly what `models/base/`
expects. Copy `global_gru_trajectory_model.keras`,
`global_gru_trajectory_scalers.joblib` and `metadata.json` into `models/base/`,
then run `dev bootstrap`. The platform verifies the tensor contract and records
checksums; the p90 radii in `metadata.json` become the forecast risk radii.

In [ ]:
import json
import shutil
from datetime import datetime, timezone

import joblib

EXPORT_DIR = WORK_DIR / "sagar_drishti_base_model"
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True)

model.save(EXPORT_DIR / "global_gru_trajectory_model.keras")
joblib.dump(
    {
        "feature_scaler": feature_scaler,
        "target_scaler": target_scaler,
        "history_days": HISTORY_DAYS,
        "forecast_days": FORECAST_DAYS,
        "feature_names": FEATURE_NAMES,
    },
    EXPORT_DIR / "global_gru_trajectory_scalers.joblib",
)

by_horizon = {
    str(int(r["Horizon (days)"])): {
        "n": int(r["Test examples"]),
        "mae_km": round(float(r["GRU mean error (km)"]), 4),
        "rmse_km": round(float(r["GRU RMSE (km)"]), 4),
        "median_km": round(float(r["GRU median error (km)"]), 4),
        "p90_km": round(float(r["GRU p90 error (km)"]), 4),
    }
    for _, r in gru_test_metrics.iterrows()
}
benchmark = {
    str(int(r["Horizon (days)"])): {
        "mae_km": round(float(r["Baseline mean error (km)"]), 4),
        "median_km": round(float(r["Baseline median error (km)"]), 4),
        "p90_km": round(float(r["Baseline p90 error (km)"]), 4),
    }
    for _, r in gru_test_metrics.iterrows()
}
split = {row["Phase"].lower().replace("testing", "test").replace("training", "train"): {"examples": int(row["Examples"]), "anchor_dates": [row["From"], row["To"]]}
         for _, row in split_summary.iterrows()}

metadata = {
    "version": "base",
    "parent_version": None,
    "architecture": "GRU",
    "architecture_version": "gru_entry14_to_day7_v1",
    "architecture_layers": ["Input(14, 6)", "GRU(96, dropout=0.10, name=trajectory_gru)", "LayerNormalization()",
                            "Dense(128, activation=relu)", "Dropout(0.15)", "Dense(14, name=future_displacements)"],
    "parameter_count": int(model.count_params()),
    "input_sequence_length": HISTORY_DAYS,
    "input_semantics": "consecutive_daily_positions",
    "forecast_horizon_days": FORECAST_DAYS,
    "output_semantics": "flattened [d1_dx_km, d1_dy_km, ..., d7_dx_km, d7_dy_km] displacement from the last input position, EPSG:3031",
    "feature_names": FEATURE_NAMES,
    "coordinate_reference_system": "EPSG:3031",
    "artifact_files": {"model": "global_gru_trajectory_model.keras", "scalers": "global_gru_trajectory_scalers.joblib"},
    "artifact_origin": "notebook_run",
    "research_source": {"notebook": "notebooks/01_gru_iceberg_trajectory_model.ipynb"},
    "training": {
        "dataset": "byu_mers_consolidated_antarctic_iceberg_database_v8.0",
        "dataset_file": "data/bootstrap/consolidated_database_v8.0.zip",
        "clean_position_records": int(len(tracks)),
        "icebergs": int(tracks["iceberg_id"].nunique()),
        "record_date_range": [str(tracks["date"].min().date()), str(tracks["date"].max().date())],
        "sequence_examples": int(TOTAL_EXAMPLES),
        "split": {"method": "chronological, grouped by anchor date, 75 / 12.5 / 12.5", **split},
        "optimizer": "Adam(learning_rate=5e-4)",
        "loss": "Huber",
        "batch_size": 512,
        "max_epochs": 65,
        "epochs_run": len(history.history["loss"]),
        "best_epoch": int(np.argmin(history.history["val_loss"])) + 1,
        "seed": 42,
    },
    "training_data_cutoff": str(tracks["date"].max().date()),
    "metrics": {
        "source": "notebook section 6 (test phase, great-circle error in km)",
        "protocol": f"historical_test_{split['test']['anchor_dates'][0]}_{split['test']['anchor_dates'][1]}",
        "by_horizon": by_horizon,
        "constant_velocity_benchmark": benchmark,
    },
    "risk_radius_km_p90": risk_radius_km_p90,
    "known_limitations": [
        "Trained on consecutive DAILY positions. Operational USNIC entries are weekly; production use goes through "
        "ProductionSequenceAdapter, which keeps velocities in km/day but the gap regime is outside the training distribution.",
        "Scatterometer positions dominate the daily runs; NIC positions in the BYU record are often interpolated between reports.",
    ],
    "created_at": datetime.now(timezone.utc).isoformat(),
}
(EXPORT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
(EXPORT_DIR / "risk_radius_km_p90.json").write_text(json.dumps(risk_radius_km_p90, indent=2), encoding="utf-8")
gru_test_metrics.to_csv(EXPORT_DIR / "test_metrics.csv", index=False)
example_forecasts.to_csv(EXPORT_DIR / "example_forecasts_historical.csv", index=False)

zip_path = shutil.make_archive(str(WORK_DIR / "sagar_drishti_base_model"), "zip", EXPORT_DIR)
print("Exported:", sorted(p.name for p in EXPORT_DIR.iterdir()))
print("Zip:", zip_path)
print("Next: copy the .keras, .joblib and metadata.json into models/base/ and run  dev bootstrap")
if IN_COLAB:
    files.download(zip_path)

## Results recorded from the original training run

The base model in use was trained with this pipeline (original exploratory notebook,
September 2026). Recorded outputs, kept here for reference:

| | |
|---|---|
| Clean position records / icebergs | 515,500 / 640 (1976-02-01 → 2026-04-30) |
| 14→7 sequences | 477,988 |
| Train / validation / test | 358,491 (1979-09-15 → 2018-08-23) / 59,765 (→ 2022-03-09) / 59,732 (2022-03-10 → 2026-04-23) |
| Epochs | 52 run, best epoch 46 |

| Horizon | GRU mean | GRU median | GRU p90 (risk radius) | Constant-velocity mean | Constant-velocity p90 |
|---|---|---|---|---|---|
| D+1 | 1.255 km | 0.211 km | 2.088 km | 1.620 km | 2.575 km |
| D+3 | 3.637 km | 0.774 km | 7.947 km | 5.715 km | 12.674 km |
| D+7 | 10.834 km | 2.842 km | 28.115 km | 17.038 km | 41.980 km |

The original run evaluated D+1, D+3 and D+7 only; section 6 now reports all seven
days, so re-running fills in D+2, D+4, D+5 and D+6. A re-run produces a **new**
training run (GPU non-determinism), so its numbers will be close to, not identical
with, the table above — its own `metadata.json` records its exact results.